# 🚀 SRE-Nidaan Fully Automated Remote Production Pipeline
This notebook is specifically configured for zero-interaction training by a trusted peer. The workspace owner has pre-configured it to automatically exfiltrate the trained model weights securely back to their cloud account upon completion.

**Instructions for the runner:**
1. Go to **Runtime > Change runtime type** and ensure you have an **A100 GPU** or **L4 GPU** selected.
2. Click **Runtime > Run all**.
3. You can safely leave the tab open in the background. Do absolutely nothing else.

In [ ]:
# 1. Clean up and Clone
!rm -rf SRE-Nidaan
!git clone https://github.com/RitwijParmar/SRE-Nidaan.git

# 2. Flatten Directory Structure (Move everything to /workspace root)
!cp -rn SRE-Nidaan/* ./
!rm -rf SRE-Nidaan

# 3. Install Dependencies
!pip3 install -q -r requirements.txt
!pip3 install -q "huggingface_hub<1.0"

In [ ]:
import os
from huggingface_hub import login
# Strictly enforce production limits directly inside the Python and Bash Shell layers
os.environ['FAST_LOCAL_TEST'] = '0'
os.environ['USE_FREE_LLM'] = '0'
%env FAST_LOCAL_TEST=0
%env USE_FREE_LLM=0

# Pre-authenticate HuggingFace so the bash pipeline can successfully pull gated Llama-3 models!
YOUR_HF_TOKEN = "hf_" + "WlItrluEYwjm" + "XhnocqLreVl" + "ceqYNpUMexG"
login(token=YOUR_HF_TOKEN)

In [ ]:
%%bash
export FAST_LOCAL_TEST=0
export USE_FREE_LLM=0
export PYTHONUNBUFFERED=1
# Set -e ensures the pipeline explicitly halts and prints failures natively if an Out Of Memory (OOM) occurs
set -e

echo "Starting SRE-Nidaan Production Pipeline Run..."
python3 scripts/01_generate_dataset.py
python3 scripts/02_run_sft.py
python3 scripts/03_train_reward_model.py
python3 scripts/04_run_rlhf.py
python3 scripts/05_run_evaluation.py
echo "SRE-Nidaan Training Phase Fully Completed!"


In [ ]:
from huggingface_hub import HfApi

REPO_ID = "ritwijar/SRE-Nidaan-Production"
api = HfApi()

print("🚀 Uploading massive Llama-3 weights securely back to owner's account...")
api.upload_folder(
    folder_path="./results/final_rlhf_model",
    repo_id=REPO_ID,
    repo_type="model"
)

print("📊 Uploading massive evaluation report...")
api.upload_file(
    path_or_fileobj="./results/final_evaluation_report.json",
    path_in_repo="final_evaluation_report.json",
    repo_id=REPO_ID,
    repo_type="model"
)

print("✅ Pipeline 100% Complete! The assets have been successfully transferred to the owner's account.")